# EGM722 Assignment - Flood Exposure Analysis

This notebook maps flooding from Hurricane Beryl (July 2024) near Houston, Texas
using Sentinel-1 SAR backscatter data.

SAR is well suited to flood mapping because radar can see through cloud cover,
which is common during storms. Flooded surfaces show up much darker than dry land
because the flat water surface bounces the radar signal away from the sensor rather than back towards it.

Run `01_data_download.ipynb` first to download the data.


In [ ]:
import numpy as np
import rasterio
import rasterio.merge
from rasterio.vrt import WarpedVRT
from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds
from rasterio.mask import mask as rio_mask
from rasterio.features import shapes
from rasterstats import zonal_stats
from shapely.geometry import box, shape, mapping
from shapely.validation import make_valid
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
from pathlib import Path
import glob


In [ ]:
# study area and settings
BBOX = (-95.8, 29.5, -94.8, 30.2) # west, south, east, north
FLOOD_THRESHOLD = 0.05 # backscatter below this = water
CRS = 'EPSG:32614' # UTM Zone 14N for Texas

raw_dir = Path('data/raw')
processed_dir = Path('data/processed')
outputs_dir = Path('outputs')
processed_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

## Helper functions

In [ ]:
def dissolve_valid(gdf, to_crs='EPSG:4326', simplify_tolerance=None):
    """Repair geometries, dissolve into a single polygon, and reproject.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Input GeoDataFrame with one or more polygon geometries.
    to_crs : str, optional
        EPSG code or proj string for the output CRS. Default 'EPSG:4326'.
    simplify_tolerance : float or None, optional
        If given, simplify geometries by this tolerance before dissolving,
        used to reduce vertex count for display.

    Returns
    -------
    geopandas.GeoDataFrame
        Single-row GeoDataFrame with a valid, dissolved geometry in to_crs.
    """
    result = gdf.copy()
    if simplify_tolerance is not None:
        result['geometry'] = result.geometry.simplify(simplify_tolerance)
    result['geometry'] = result.geometry.apply(make_valid)
    result = result[~result.geometry.is_empty]
    return result.dissolve().to_crs(to_crs)


def mosaic_and_clip(vv_files, bbox, target_crs):
    """Mosaic Sentinel-1 tiles and clip to a bounding box.

    Handles tiles in different CRSs by warping each to target_crs
    before merging.

    Parameters
    ----------
    vv_files : list of str or Path
        Paths to the VV-polarisation GeoTIFF tiles to mosaic.
    bbox : tuple of float
        Bounding box as (west, south, east, north) in WGS84 (EPSG:4326).
    target_crs : CRS or str
        Output CRS for the mosaic, usually the CRS of the first tile.

    Returns
    -------
    mosaic : numpy.ndarray
        3-D array (bands, rows, cols) of backscatter values.
    mosaic_transform : affine.Affine
        Pixel to coordinate transform for the mosaic.
    meta : dict
        Rasterio metadata dict for writing the output file.
    """
    datasets = [rasterio.open(f) for f in vv_files]
    vrts = []
    for ds in datasets:
        if ds.crs != target_crs:
            vrts.append(WarpedVRT(ds, crs=target_crs))
        else:
            vrts.append(ds)

    # transform_bounds converts the WGS84 bbox into the raster CRS
    clip_bounds = transform_bounds('EPSG:4326', target_crs, *bbox)
    mosaic, mosaic_transform = rasterio.merge.merge(vrts, bounds=clip_bounds)

    meta = datasets[0].meta.copy()
    meta.update({'crs': target_crs, 'height': mosaic.shape[1],
                 'width': mosaic.shape[2], 'transform': mosaic_transform})
    for ds in datasets:
        ds.close()
    return mosaic, mosaic_transform, meta


def vectorise_flood_mask(flood_mask_path, min_pixels=4):
    """Convert a binary flood mask raster to a GeoDataFrame of polygons.

    Traces polygon outlines around flooded pixels, removes patches smaller
    than min_pixels, and applies a small simplification and buffer(0)
    to clean up edges.

    Parameters
    ----------
    flood_mask_path : str or Path
        Path to the flood mask GeoTIFF (1 = flooded, 0 = dry).
    min_pixels : int, optional
        Minimum patch size in pixels. Smaller patches are dropped as noise.
        Default is 4.

    Returns
    -------
    geopandas.GeoDataFrame
        GeoDataFrame of flood polygons in the raster's CRS.
    """
    with rasterio.open(flood_mask_path) as src:
        data = src.read(1)
        flood_crs = src.crs
        t = src.transform
        pixel_area_m2 = abs(src.transform.a * src.transform.e)

    # shapes() traces outlines around groups of identical pixels
    flood_pixels_arr = (data == 1).astype(np.uint8)
    raw_geoms = [shape(geom) for geom, _ in shapes(flood_pixels_arr, mask=flood_pixels_arr, transform=t)]

    min_area = pixel_area_m2 * min_pixels
    clean_geoms = [g for g in raw_geoms if g.area >= min_area]

    flood_gdf = gpd.GeoDataFrame(geometry=clean_geoms, crs=flood_crs)
    # buffer(0) fixes invalid geometries that can come up after simplification
    flood_gdf['geometry'] = flood_gdf.geometry.simplify(30).buffer(0)
    flood_gdf = flood_gdf[~flood_gdf.geometry.is_empty].reset_index(drop=True)
    return flood_gdf


## 1. Load the SAR data

The downloaded data is OPERA L2 RTC-S1: Sentinel-1 backscatter that has been
radiometrically terrain-corrected (RTC) to remove distortions caused by topography.
VV polarisation (vertical transmit, vertical receive) is used as it gives the clearest
contrast between open water and land.

In [ ]:
# find downloaded VV polarisation GeoTIFFs
vv_files = glob.glob(str(raw_dir / '**/*VV*.tif'), recursive=True)
vv_files += glob.glob(str(raw_dir / '*VV*.tif'))
print(f'Found {len(vv_files)} VV file(s)')

In [ ]:
target_crs = rasterio.open(vv_files[0]).crs
mosaic, mosaic_transform, meta = mosaic_and_clip(vv_files, BBOX, target_crs)

clipped_path = processed_dir / 'sar_clipped.tif'
with rasterio.open(clipped_path, 'w', **meta) as dst:
    dst.write(mosaic)
print(f'SAR data loaded and clipped: {clipped_path}')


In [ ]:
# reproject to UTM Zone 14N - projected CRS for Texas so distances are in metres
with rasterio.open(clipped_path) as src:
    transform, width, height = calculate_default_transform(
        src.crs, CRS, src.width, src.height, *src.bounds)
    meta = src.meta.copy()
    meta.update({'crs': CRS, 'transform': transform, 'width': width, 'height': height})
    reprojected_path = processed_dir / 'sar_utm.tif'
    with rasterio.open(reprojected_path, 'w', **meta) as dst:
        reproject(source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
                  src_crs=src.crs, dst_crs=CRS, resampling=Resampling.bilinear)
print(f'Reprojected: {reprojected_path}')


## 2. Create the flood mask

Flood detection works by thresholding the backscatter values. Open water returns
very low backscatter ~below 0.05 because the flat water
surface reflects radar energy away from the sensor rather than back towards it.

The histogram below shows the distribution of backscatter values across the study area.
The red dashed line marks the flood threshold, pixels to the left are classified as water.

In [ ]:
# plot backscatter histogram to check the threshold makes sense
with rasterio.open(reprojected_path) as src:
    data = src.read(1).astype(float)
    nodata = src.nodata
    if nodata is not None:
        data[data == nodata] = np.nan

plt.figure(figsize=(8, 4))
plt.hist(data[~np.isnan(data)].ravel(), bins=100, range=(0, 0.5), color='steelblue', edgecolor='none')
plt.axvline(FLOOD_THRESHOLD, color='red', linestyle='--', label=f'Threshold = {FLOOD_THRESHOLD}')
plt.xlabel('Backscatter (linear)')
plt.ylabel('Pixel count')
plt.title('SAR backscatter distribution')
plt.legend()
plt.tight_layout()
plt.show()

The peak at the low end of the distribution represents water bodies (rivers, lakes, and
flooded areas). The peak to the right represents land. If the threshold line sits
between these two peaks the classification is well calibrated.
If there were too many or too few pixels are being classified as flooded, you could adjust `FLOOD_THRESHOLD`
in the configuration cell above and re-run.

In [ ]:
# classify pixels below the threshold as flooded
with rasterio.open(reprojected_path) as src:
    data = src.read(1).astype(float)
    nodata = src.nodata
    meta = src.meta.copy()

if nodata is not None:
    data[data == nodata] = np.nan

flood = (data < FLOOD_THRESHOLD).astype(np.uint8)
flood[np.isnan(data)] = 0

meta.update({'dtype': 'uint8', 'nodata': 255})
flood_mask_path = processed_dir / 'flood_mask.tif'
with rasterio.open(flood_mask_path, 'w', **meta) as dst:
    dst.write(flood, 1)

flood_pixels = int(flood.sum())
with rasterio.open(flood_mask_path) as src:
    pixel_area_m2 = abs(src.transform.a * src.transform.e)
flood_area_km2 = flood_pixels * pixel_area_m2 / 1e6
print(f'Flooded pixels: {flood_pixels}  (~{flood_area_km2:.1f} km²)')

In [ ]:
flood_gdf = vectorise_flood_mask(flood_mask_path)
print(f'Flood extent: {len(flood_gdf)} polygons')


The flood mask has been converted to vector polygons, to make it easier to overlay with
other datasets and calculate some statistics. The total flooded area estimate above includes water bodies like rivers and reservoirs as well as storm-induced flooding - to calculate just the flood area I would need to do a pre flood analysis too.

## 3. Population exposure

The WorldPop gridded population dataset provides an estimate of how many people
live in each 1 km² grid cell. Summing population within the flood extent gives
an estimate of the number of people exposed to flooding.

In [ ]:
# clip the WorldPop raster to the study area
worldpop_raw = raw_dir / 'worldpop_usa_2020_1km.tif'
worldpop_clipped = processed_dir / 'worldpop_clipped.tif'

with rasterio.open(worldpop_raw) as src:
    clip_geom = [mapping(box(*BBOX))]
    out_image, out_transform = rio_mask(src, clip_geom, crop=True, nodata=-99999)
    meta = src.meta.copy()
    meta.update({'height': out_image.shape[1], 'width': out_image.shape[2],
                 'transform': out_transform, 'nodata': -99999})

with rasterio.open(worldpop_clipped, 'w', **meta) as dst:
    dst.write(out_image)
print(f'WorldPop clipped: {worldpop_clipped}')


In [ ]:
# dissolve all flood polygons into one shape before running zonal_stats
# much faster than passing hundreds of individual polygons
flood_4326 = dissolve_valid(flood_gdf)

stats = zonal_stats(flood_4326, str(worldpop_clipped), stats=['sum'], nodata=-99999)
exposed_population = int(sum(s['sum'] for s in stats if s['sum'] and s['sum'] > 0))
print(f'Estimated population exposed to flooding: {exposed_population:,}')


The population figure above is an estimate based on 2020 WorldPop data at 1 km resolution.
It represents the number of people who lived in areas mapped as flooded - but not necessarily
people who were directly affected.
The 1 km resolution means small flooded areas (narrower than ~1 km) may be under-counted.

## 4. Map

The interactive map below shows the flood extent (blue).
Use the layer control in the top right to toggle layers on and off.

In [ ]:
# interactive map centred on Houston
m = folium.Map(location=[29.75, -95.5], zoom_start=9, tiles='CartoDB positron')

# simplify polygons just for display - rendering every polygon at full
# resolution in the browser is very slow
if len(flood_gdf) > 0:
    flood_display = dissolve_valid(flood_gdf, simplify_tolerance=100)
    flood_display.explore(m=m, color='blue', style_kwds={'fillOpacity': 0.4, 'weight': 0}, name='Flood extent')

folium.LayerControl(collapsed=False).add_to(m)
m.save(str(outputs_dir / 'flood_map.html'))
print('Map saved to outputs/flood_map.html')
m


## 5. Summary


In [ ]:
print('Flood threshold:', FLOOD_THRESHOLD, '(linear backscatter)')
print('Estimated flood area:', round(flood_area_km2, 1), 'km2')
print('Estimated population exposed:', f'{exposed_population:,}')

### Discussion

Hurricane Beryl made landfall as a Category 1 hurricane near Matagorda Bay on 8 July 2024
and moved inland over the Houston metropolitan area, bringing heavy rainfall and storm surge.

The flood mapped here is based on a single post-storm Sentinel-1 acquisition.
The OPERA RTC product provides 30 m resolution backscatter, which can map major flood extents
but may miss narrow flooded streets or small water bodies.

The population exposure estimate uses WorldPop 2020 data at 1 km resolution. This gives
a rough indication of how many people were in the affected area.
